# Pinky-Pro QR / ArUco 검출 테스트

Pinky-Pro 카메라 영상에서 QR 데이터와 ArUco 마커를 검출하기 위한 노트북입니다.

In [ ]:
import cv2
import numpy as np

if not hasattr(cv2, 'aruco'):
    raise ImportError('OpenCV ArUco 모듈이 필요합니다: pip install opencv-contrib-python')


def make_aruco_detector(dictionary_id):
    dictionary = cv2.aruco.getPredefinedDictionary(dictionary_id)
    if hasattr(cv2.aruco, 'ArucoDetector'):
        parameters = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(dictionary, parameters)
        return detector.detectMarkers
    parameters = cv2.aruco.DetectorParameters_create()
    return lambda image: cv2.aruco.detectMarkers(
        image, dictionary, parameters=parameters
    )


def _polygon_points(points):
    return np.asarray(points, dtype=np.float32).reshape(-1, 2)


def detect_qr_codes(frame):
    annotated = frame.copy()
    detector = cv2.QRCodeDetector()
    results = []

    try:
        found, decoded_info, points, _ = detector.detectAndDecodeMulti(frame)
    except (AttributeError, cv2.error):
        found, decoded_info, points = False, (), None

    if found and points is not None:
        candidates = zip(decoded_info, points)
    else:
        data, single_points, _ = detector.detectAndDecode(frame)
        candidates = [(data, single_points)] if single_points is not None else []

    for data, qr_points in candidates:
        polygon = _polygon_points(qr_points)
        if not data or len(polygon) != 4:
            continue
        integer_polygon = np.round(polygon).astype(np.int32)
        cv2.polylines(annotated, [integer_polygon], True, (0, 255, 0), 2)
        x, y = integer_polygon[0]
        cv2.putText(
            annotated, f'QR: {data}', (int(x), max(20, int(y) - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
        )
        results.append({'data': data, 'corners': polygon.tolist()})
    return annotated, results


def _estimate_square_pose(corners, marker_size_m, camera_matrix, dist_coeffs):
    half = marker_size_m / 2.0
    object_points = np.array(
        [[-half, half, 0.0], [half, half, 0.0],
         [half, -half, 0.0], [-half, -half, 0.0]],
        dtype=np.float32,
    )
    success, rvec, tvec = cv2.solvePnP(
        object_points, corners.astype(np.float32), camera_matrix, dist_coeffs,
        flags=cv2.SOLVEPNP_IPPE_SQUARE,
    )
    if not success:
        return None, None
    return rvec.reshape(3), tvec.reshape(3)


def detect_aruco_markers(
    frame, dictionary_specs, camera_matrix=None, dist_coeffs=None
):
    annotated = frame.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    results = []
    seen_centers = []

    for dictionary_name, dictionary_id, marker_size_m in dictionary_specs:
        corners_list, ids, _ = make_aruco_detector(dictionary_id)(gray)
        if ids is None:
            continue
        for marker_corners, marker_id in zip(corners_list, ids.flatten()):
            corners = _polygon_points(marker_corners)
            center = corners.mean(axis=0)
            if any(np.linalg.norm(center - old) < 5.0 for old in seen_centers):
                continue
            seen_centers.append(center)
            rvec = tvec = None
            if camera_matrix is not None and dist_coeffs is not None:
                rvec, tvec = _estimate_square_pose(
                    corners, marker_size_m, camera_matrix, dist_coeffs
                )
            polygon = np.round(corners).astype(np.int32)
            cv2.polylines(annotated, [polygon], True, (255, 0, 0), 2)
            label = f'{dictionary_name} ID={int(marker_id)}'
            if tvec is not None:
                label += f' xyz=({tvec[0]:.3f}, {tvec[1]:.3f}, {tvec[2]:.3f})m'
                cv2.drawFrameAxes(
                    annotated, camera_matrix, dist_coeffs,
                    rvec.reshape(3, 1), tvec.reshape(3, 1), marker_size_m * 0.5
                )
            x, y = polygon[0]
            cv2.putText(
                annotated, label, (int(x), max(20, int(y) - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 2
            )
            results.append({
                'dictionary': dictionary_name,
                'id': int(marker_id),
                'corners': corners.tolist(),
                'rvec': None if rvec is None else rvec.tolist(),
                'tvec_m': None if tvec is None else tvec.tolist(),
            })
    return annotated, results
